In [1]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnableBranch, RunnablePassthrough

In [2]:
def retrieve_sql(query: str) -> str:
    return f"\n SQL retrieval for: {query}"

def retrieve_pdf(query: str) -> str:
    return f"\n PDF retrieval for: {query}"

def retrieve_general(query: str) -> str:
    return f"\n General retrieval for: {query}"

In [3]:
branch = RunnableBranch(
    (lambda x: "financial" in x["query"].lower(),
     RunnablePassthrough.assign(context=lambda x: retrieve_sql(x["query"]))),
    (lambda x: "technical" in x["query"].lower(),
     RunnablePassthrough.assign(context=lambda x: retrieve_pdf(x["query"]))),
    # Default branch: just the runnable
    RunnablePassthrough.assign(context=lambda x: retrieve_general(x["query"]))
)

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use the provided context."),
    ("user", "Context:\n{context}\n\nQuery:\n{query}")
])

In [5]:
llm = ChatOpenAI(model="gpt-4", temperature=0)

rag_chain = branch | prompt | llm

In [6]:
print(rag_chain.invoke({"query": "financial report"}))
#print(rag_chain.invoke({"query": "technical manual"}))
print(rag_chain.invoke({"query": "random topic"}))

content='SELECT * FROM Financial_Reports;' response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 35, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-9cb9d1e1-2666-4a4f-a122-dccef8631fea-0'
content="Sure, how about we discuss the impact of technology on society? It's a broad topic that covers many aspects such as communication, education, healthcare, and even entertainment." response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 35, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 